# 07 — Random Forest
Review the validation-tuned forest already saved in Phase 4 and recompute its held-out predictions. No trees are fitted, no hyperparameters are changed, and no frozen file is rewritten.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import numpy as np
import pandas as pd
from ml.features import FEATURE_COLUMNS
from ml.phase4_models import load_predictor, metrics, predictions_agree
from ml.train_phase4 import read_partition, verify_manifest
manifest = json.loads(Path('ml/data/phase3_dataset_manifest.json').read_text(encoding='utf-8'))
results = json.loads(Path('ml/artifacts/phase4/model_results.json').read_text(encoding='utf-8'))
lock = json.loads(Path('ml/artifacts/phase4/validation_selection.json').read_text(encoding='utf-8'))
verify_manifest(manifest)
bundle = load_predictor(results['artifact']['path'], results['artifact']['sha256'])
model = bundle.models['random_forest']
print('Interpreter:', sys.executable)

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe


## Saved configuration and mathematics
ŷ(x) is the average of 100 tree leaf predictions. Bootstrap samples and feature subsampling reduce tree correlation. No scaling is required. Frozen validation tuning compared depth {8,12} × minimum leaf {10,30}; other parameters were fixed. Validation MAE/RMSE determined the stored forest configuration before test access.
Parallel tree summation can differ by floating roundoff. Reload comparison uses the existing absolute 1e−12 tolerance with zero relative tolerance. This does not permit meaningful score changes.

In [2]:
chosen = lock['tuning']['chosen_forest_parameters']
assert all(model.get_params()[key] == value for key, value in chosen.items())
assert len(model.estimators_) == 100 and model.n_features_in_ == len(FEATURE_COLUMNS)
display(pd.Series(chosen, name='Saved parameters').to_frame())
display(pd.DataFrame([{'candidate': c['candidate'], 'max_depth': c['parameters']['max_depth'], 'min_samples_leaf': c['parameters']['min_samples_leaf'], **c['validation']['clipped']} for c in lock['tuning']['forest_candidates']]))
importance = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS, name='impurity_importance')
display(importance.sort_values(ascending=False).head(10).to_frame())
print('Importance is descriptive and biased by correlated predictors; it is not causal attribution.')

,Saved parameters
n_estimators,100
max_depth,8
min_samples_leaf,30
max_features,0.8
max_samples,0.8
bootstrap,True
random_state,42
n_jobs,2


,candidate,max_depth,min_samples_leaf,mae,rmse,r2
0,0,8,10,0.445964,0.695580,0.948771
1,1,8,30,0.436999,0.680876,0.950914
2,2,12,10,0.484498,0.729381,0.943671
3,3,12,30,0.458959,0.700878,0.947987


,impurity_importance
total_occupied,0.780602
occupied_lag_1,0.155499
occupied_mean_5,0.030479
occupied_lag_5,0.015678
time_sin,0.006210
time_cos,0.001899
occupied_mean_1440,0.001364
occupied_mean_60,0.001188
occupied_lag_10080,0.001186
occupied_std_60,0.001096


Importance is descriptive and biased by correlated predictors; it is not causal attribution.


In [3]:
rows = []
for split in ('validation', 'test'):
    x, y, identity = read_partition(manifest, split)
    assert identity == results['partitions'][split]
    raw_prediction = model.predict(x)
    measured = metrics(y, raw_prediction)
    frozen = results[split]['random_forest']
    assert measured['rows'] == frozen['rows']
    assert measured['clipped_count'] == frozen['clipped_count'] == 0
    for kind in ('raw', 'clipped'):
        for key in ('mae', 'rmse', 'r2'):
            np.testing.assert_allclose(measured[kind][key], frozen[kind][key], rtol=0, atol=1e-12)
    bounded = bundle.predict(pd.DataFrame(x, columns=FEATURE_COLUMNS, copy=False), 'random_forest')
    assert predictions_agree(bounded, np.clip(raw_prediction, 0, 12))
    assert np.isfinite(bounded).all() and ((bounded >= 0) & (bounded <= 12)).all()
    rows.append({'split': split, 'rows': len(y), **measured['clipped'], 'clipped_count': measured['clipped_count'], 'origin_sha256': identity['origin_sha256']})
    if split == 'test':
        display(pd.DataFrame({'actual_t_plus_15': y[:8], 'prediction': bounded[:8]}))
display(pd.DataFrame(rows))
print('PASS: same frozen rows, score agreement and bounded saved-model reload predictions')

,actual_t_plus_15,prediction
0,3.0,2.934256
1,3.0,2.939143
2,3.0,2.939143
3,3.0,2.951806
4,3.0,2.951806
5,3.0,2.951806
6,3.0,2.947590
7,3.0,2.947590


,split,rows,mae,rmse,r2,clipped_count,origin_sha256
0,validation,10577,0.436999,0.680876,0.950914,0,6c28dca1155df5f02c545ae508f03e006079183464b664...
1,test,10578,0.394675,0.641002,0.946445,0,79f277c74ba6bd39467ba2bf0f72e0312f6795028c6d6a...


PASS: same frozen rows, score agreement and bounded saved-model reload predictions


## Interpretation and limits
The tuned forest does not beat persistence on the frozen validation MAE criterion. It remains available in the comparison bundle; no test-based retuning is allowed. Predictions are fractional expectations in [0,12], not per-slot state predictions. Synthetic demand is not a substitute for real operational evaluation.